## Imports

In [46]:
import numpy as np
import pandas as pd
import os
import json
import re
from sklearn.model_selection import train_test_split
from huggingface_hub import HfApi
from datasets import Dataset

In [105]:
from datasets import load_dataset

In [5]:
from huggingface_hub import login

In [40]:
login()

In [4]:
pip install --upgrade pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install --upgrade fsspec

In [22]:
def normalize_sentence_case(text):
    # Define a function to capitalize the first letter of a sentence
    def capitalize_first_letter(match):
        return match.group(1) + match.group(2).upper()
    
    # Regular expression to match the beginning of a sentence
    sentence_start_re = re.compile(r'([.!?]\s+|^)(\w)')
    
    # Lowercase the entire text first
    text = text.lower()
    
    # Removing any brackets
    text = re.sub(r'\([^)]*\)', '', text)
    # Removing double spaces
    text = re.sub(r'\s+', ' ', text)
    # Use the sub method to apply the capitalize_first_letter function to each match
    normalized_text = sentence_start_re.sub(capitalize_first_letter, text)
    
    return normalized_text

# Example usage
weird_casing_text = "tHis iS a sEntEnCe. here IS anothEr oNe!  why Is tHis HaPPening (idk)?"
normalized_text = normalize_sentence_case(weird_casing_text)
print(normalized_text)

This is a sentence. Here is another one! Why is this happening ?


## Food LoRA Dataset

In [ ]:
#Conversion to dataframe
food_df = pd.read_json(TRAIN_DS, lines=True) #Replace this with new huggingface models

In [54]:
animal_nutrition_unformatted_df = pd.read_parquet("hf://datasets/A2H0H0R1/Animal-nutrition/data/train-00000-of-00001.parquet")

In [63]:
#Conversion to prompt, response format
df_list = []

for i in range(len(animal_nutrition_unformatted_df)):
    row = animal_nutrition_unformatted_df.iloc[i,:].values[0]
    question_index = row.find("Question:")
    answer_index = row.find("Answer:")
    #print(normalize_sentence_case(row[question_index + 10: answer_index]))
    #print(normalize_sentence_case(row[answer_index + 8:]))
    df_list.append([normalize_sentence_case(row[question_index + 10: answer_index]),\
               normalize_sentence_case(row[answer_index + 8:])])

In [64]:
animal_nutrition_df = pd.DataFrame(df_list, columns = ['prompt', 'response'])

In [69]:
animal_nutrition_df.iloc[0,1]

'The main components of foods, plants, and animals include water, dry matter, carbohydrates, lipids, proteins, nucleic acids, organic acids, vitamins, and minerals. Water is a crucial component of all living organisms and makes up the majority of the weight of most foods, plants, and animals. It plays a vital role in maintaining cellular structure and function, regulating body temperature, and facilitating metabolic processes. Dry matter refers to the solid portion of foods, plants, and animals that remains after removing all moisture. Dry matter components include carbohydrates, lipids, proteins, nucleic acids, organic acids, vitamins, and minerals. These components provide essential nutrients for the growth, maintenance, and repair of cells and tissues in animals. Carbohydrates are one of the three macronutrients found in foods, plants, and animals. They are the primary source of energy for the body and are composed of glucose, fructose, and galactose. Carbohydrates are also importan

In [77]:
nutrition_df = pd.read_parquet("hf://datasets/Tom158/Nutritional-LLama/data/train-00000-of-00001.parquet")

In [78]:
nutrition_df.drop(columns = ['System', 'text'], inplace = True)

In [79]:
nutrition_df.rename(columns = {'User': 'prompt', 'Nutritionist': 'response'}, inplace = True)

In [92]:
nutrition_df['prompt'] = nutrition_df['prompt'].apply(normalize_sentence_case)

In [93]:
nutrition_df['response'] = nutrition_df['response'].apply(normalize_sentence_case)

In [95]:
food_df = pd.concat([animal_nutrition_df, nutrition_df], ignore_index=True)

In [96]:
food_df['prompt'] = food_df['prompt'].apply(normalize_sentence_case)

In [97]:
food_df['response'] = food_df['response'].apply(normalize_sentence_case)

In [98]:
food_df['prompt']

0       What are the main components of foods, plants,...
1       How do the various classes of animals, such as...
2       Considering the role of plants in animal nutri...
3       What is the significance of water in the conte...
4       How do the different classes of chemical subst...
                              ...                        
6510    Should i be worried about consuming mayonnaise...
6511    I've been trying to cut down on sugar but i lo...
6512    Should i avoid eating leg roasted because of m...
6513    I recently indulged in a slice of lemon mering...
6514    Should i be worried about eating cocoa conside...
Name: prompt, Length: 6515, dtype: object

In [99]:
#Train, test, val split
train_food_df, temp_df = train_test_split(food_df, test_size=0.2, random_state=42)
val_food_df, test_food_df = train_test_split(temp_df, test_size=0.25, random_state=42)

In [102]:
train_food_dataset = Dataset.from_pandas(train_food_df)
val_food_dataset = Dataset.from_pandas(val_food_df)
test_food_dataset = Dataset.from_pandas(test_food_df)

In [100]:
HfApi().create_repo(repo_id="GaurangPP/Food", repo_type="dataset")

RepoUrl('https://huggingface.co/datasets/GaurangPP/Food', endpoint='https://huggingface.co', repo_type='dataset', repo_id='GaurangPP/Food')

In [103]:
train_food_dataset.push_to_hub("GaurangPP/Food", private=False, split = "train")
val_food_dataset.push_to_hub("GaurangPP/Food", private=False, split = "validation")
test_food_dataset.push_to_hub("GaurangPP/Food", private=False, split = "test")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/6 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/358 [00:00<?, ?B/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/473 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/GaurangPP/Food/commit/6aa78083cb444d4641663edb5dbd8bf022755472', commit_message='Upload dataset', commit_description='', oid='6aa78083cb444d4641663edb5dbd8bf022755472', pr_url=None, pr_revision=None, pr_num=None)

## Exercise LoRA Dataset

In [106]:
exercise_one_ds = load_dataset("ayush0639/fitnessguru")

separated_records.txt:   0%|          | 0.00/396k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4296 [00:00<?, ? examples/s]

In [131]:
print(exercise_one_ds['train'][6]['text'])

 connect with others reach out to family friends colleagues and online support groups


In [133]:
exercise_one_list = []

for i in range(len(exercise_one_ds['train'])):
    #Prompt and response list
    qa = []
    #Getting text from dataframe
    text = exercise_one_ds['train'][i]['text']
    #If human in text then collect prompt data
    if "Human:" in text:
        qa.append(text[7:])
        response = ""
        #Collect response data
        for j in range(i+1,len(exercise_one_ds['train'])):
            response_text = exercise_one_ds['train'][j]['text']
            #Add to response string
            if "Assistant:" in response_text:
                response += response_text[11:]
            #Exit loop and update outer loop variable
            elif "Human:" in response_text:
                i = j - 1
                break
            #Continue to append to response string
            else:
                response += " " + response_text
        qa.append(response)
        exercise_one_list.append(qa)

In [137]:
exercise_one_list[2]

['how can i incorporate regular movement and physical activity into my day even if i have a busy schedule',
 'take the stairs instead of the elevator whenever possible  break up your day with short walks throughout the day  take a few minutes from your break to do some simple stretching exercises  stand while youre talking on the phone or reading emails  use an exercise band or do simple bodyweight exercises at your desk  park further away from your destination so you have to walk a few extra minutes  take a yoga or pilates class when you have free time  organize a group of friends to do weekly hikes or ']

In [138]:
len(exercise_one_list)

928

In [139]:
exercise_one_df = pd.DataFrame(exercise_one_list, columns = ['prompt', 'response'])

In [140]:
exercise_one_df

,prompt,response
0,can you recommend effective ab exercises,planks bicycle crunches and leg raises are gre...
1,what are some effective strategies for managin...,make time for relaxation take time to catch yo...
2,how can i incorporate regular movement and phy...,take the stairs instead of the elevator whenev...
3,are there any specific strategies for maintain...,eat a healthy diet eating a healthy diet rich ...
4,how can i manage stress and maintain a healthy...,manage your time wisely prioritize your tasks ...
...,...,...
923,what are the benefits of interval training com...,interval training such as hiit increases poste...
924,how can individuals stay motivated during plat...,setting nonscale goals focusing on progress be...
925,are there any potential side effects or risks ...,while supplements can be beneficial when used ...
926,what are some indicators that individuals may ...,persistent weight plateaus extreme fatigue moo...


In [145]:
exercise_one_df.iloc[927,1]

'emphasizing balance flexibility and selfcompassion in dietary choices and exercise routines can promote a positive mindset and sustainable habits for longterm fat loss and overall wellbeing '

In [146]:
splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
exercise_two_df_train = pd.read_parquet("hf://datasets/ramachaitanya22/mental_health_and_fitness_data/" + splits["train"])
exercise_two_df_test = pd.read_parquet("hf://datasets/ramachaitanya22/mental_health_and_fitness_data/" + splits["test"])

In [147]:
exercise_two_df = pd.concat([exercise_two_df_train, exercise_two_df_test], ignore_index=True)

In [148]:
exercise_two_df.rename(columns = {'Human': 'prompt', 'Assistant': 'response'}, inplace = True)

In [149]:
exercise_two_df

,prompt,response
0,can you provide tips for maintaining fitness w...,fitness on the go pack resistance bands explor...
1,I start counseling/therapy in a few days (I'm ...,"Lots of people do cry in session, but your the..."
2,How do you know you have the right therapist f...,"This is a really important question, because y..."
3,Every time I send a message to someone or a gr...,Sorry to hear your friends aren't responding t...
4,I have so many issues to address. I have a his...,You do not have too many issues to address in ...
...,...,...
4435,My husband and I got into a huge dispute. He s...,"In any relationship, including marriage, each ..."
4436,I have PTSD. The side effects are really bad a...,Slowly is the rate at which you'll get back yo...
4437,tell be about singlearm cable front raise,the singlearm cable front raise is an isolatio...
4438,"I was born a girl, but I want to be a boy. Bec...",Desire to inhabit the opposite sex's body deri...


In [150]:
exercise_df = pd.concat([exercise_one_df, exercise_two_df], ignore_index=True)

In [151]:
exercise_df['prompt'] = exercise_df['prompt'].apply(normalize_sentence_case)

In [152]:
exercise_df['response'] = exercise_df['response'].apply(normalize_sentence_case)

In [154]:
train_exercise_df, temp_df = train_test_split(exercise_df, test_size=0.2, random_state=42)
val_exercise_df, test_exercise_df = train_test_split(temp_df, test_size=0.25, random_state=42)

In [155]:
HfApi().create_repo(repo_id="GaurangPP/Exercise", repo_type="dataset")

RepoUrl('https://huggingface.co/datasets/GaurangPP/Exercise', endpoint='https://huggingface.co', repo_type='dataset', repo_id='GaurangPP/Exercise')

In [156]:
train_exercise_dataset = Dataset.from_pandas(train_exercise_df)
val_exercise_dataset = Dataset.from_pandas(val_exercise_df)
test_exercise_dataset = Dataset.from_pandas(test_exercise_df)

In [157]:
train_exercise_dataset.push_to_hub("GaurangPP/Exercise", private=False, split = "train")
val_exercise_dataset.push_to_hub("GaurangPP/Exercise", private=False, split = "validation")
test_exercise_dataset.push_to_hub("GaurangPP/Exercise", private=False, split = "test")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/358 [00:00<?, ?B/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/473 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/GaurangPP/Exercise/commit/bdbb17a7fea3257098cf7233395429a9a73ed98b', commit_message='Upload dataset', commit_description='', oid='bdbb17a7fea3257098cf7233395429a9a73ed98b', pr_url=None, pr_revision=None, pr_num=None)

## Code LoRA Dataset